Colab things:

In [2]:
import shutil

# Remove the folder
folder_path = '/content/shapenet-core-seg'

try:
    # shutil.rmtree(folder_path)
    print(f"Successfully deleted: {folder_path}")
except Exception as e:
    print(f"Error: {e}")

Successfully deleted: /content/shapenet-core-seg


In [3]:
import os
os.listdir()

file_path = '/content/shapenet-core-seg/benchmark.zip'
print(f"File size: {os.path.getsize(file_path) / (1024*1024):.2f} MB")

File size: 22.77 MB


In [4]:
from genericpath import exists
import zipfile
import os

zip_file_path = '/content/shapenet-core-seg/benchmark.zip'
extract_dir = '/content/shapenet-core-seg/'

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"'{zip_file_path}' unzipped to '{extract_dir}'")
print(os.listdir(extract_dir))


'/content/shapenet-core-seg/benchmark.zip' unzipped to '/content/shapenet-core-seg/'
['benchmark.zip', 'Shapenet_benchmark_sample']


In [5]:
from genericpath import exists
import zipfile
import os

zip_file_path = '/content/lidar-od-scripts.zip'
extract_dir = '/content/lidar-od-scripts/'

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"'{zip_file_path}' unzipped to '{extract_dir}'")
print(os.listdir(extract_dir))

'/content/lidar-od-scripts.zip' unzipped to '/content/lidar-od-scripts/'
['AxisAlignedTargetAssigner_CPU.pkl', 'gpuVersion', 'AxisAlignedTargetAssigner_GPU.pkl']


In [6]:
import os
import sys

# 1. Walk through the folder to find where 'visual_utils.py' is hiding
search_dir = '/content/lidar-od-scripts/'
found = False

print("Searching for visual_utils.py...")

for root, dirs, files in os.walk(search_dir):
    if 'visual_utils.py' in files:
        # We found it! Add THIS specific folder to the system path
        sys.path.append(root)
        print(f"✅ Found it in: {root}")
        print(f"✅ Added {root} to Python path.")
        found = True
        break

if not found:
    print("❌ Error: visual_utils.py was not found. The zip might be empty or structure is different.")
else:
    # 2. Now try the import again
    try:
        from visual_utils import plot_pc_data3d, plot_bboxes_3d
        print("✅ Import successful! You can now use plot_pc_data3d.")
    except ImportError as e:
        print(f"❌ Import still failed: {e}")

Searching for visual_utils.py...
✅ Found it in: /content/lidar-od-scripts/gpuVersion/gpuVersion
✅ Added /content/lidar-od-scripts/gpuVersion/gpuVersion to Python path.
✅ Import successful! You can now use plot_pc_data3d.


Set the json files to split the data for train, validate and test

In [7]:
import os
import json
import random
import glob

# Ensure this points to your actual data root
DATA_ROOT = './shapenet-core-seg/Shapenet_benchmark_sample/'

def generate_new_splits(root_dir):
    print(f"Scanning files in {root_dir}...")

    # Initialize lists to hold the final split data
    final_train = []
    final_val = []
    final_test = []

    # Map folder IDs to Class Names
    class_map = {
        '02691156': 'Airplane', '02773838': 'Bag', '02954340': 'Cap', '02958343': 'Car',
        '03001627': 'Chair', '03261776': 'Earphone', '03467517': 'Guitar', '03624134': 'Knife',
        '03636649': 'Lamp', '03642806': 'Laptop', '03790512': 'Motorbike', '03797390': 'Mug',
        '03948459': 'Pistol', '04099429': 'Rocket', '04225987': 'Skateboard', '04379243': 'Table'
    }

    # Get list of all folders in the root directory
    folders = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]

    # Iterate over each category folder separately
    for folder_id in folders:
        # Skip folders not in our map
        if folder_id not in class_map:
            continue

        class_name = class_map[folder_id]
        class_id = list(class_map.keys()).index(folder_id)

        # List to hold all valid samples for THIS specific category
        category_samples = []

        # Check for points directory
        points_dir = os.path.join(root_dir, folder_id, 'points')
        if not os.path.exists(points_dir):
            continue

        files = glob.glob(os.path.join(points_dir, "*.npy"))

        for file_path in files:
            filename = os.path.basename(file_path)
            rel_point_path = f"{folder_id}/points/{filename}"

            # Construct label path
            seg_filename = filename.replace('.npy', '.seg')
            rel_seg_path = f"{folder_id}/points_label/{seg_filename}"
            full_seg_path = os.path.join(root_dir, folder_id, 'points_label', seg_filename)

            # Only add if both point and label files exist
            if os.path.exists(full_seg_path):
                category_samples.append([class_id, class_name, rel_point_path, rel_seg_path])

        # --- KEY CHANGE: Split per category ---
        total_cat = len(category_samples)
        if total_cat == 0:
            continue

        # Shuffle only this category's samples
        random.shuffle(category_samples)

        # Calculate split indices (70% train, 10% val, rest test)
        n_train = int(total_cat * 0.7)
        n_val = int(total_cat * 0.1)

        # Append to the main lists
        final_train.extend(category_samples[:n_train])
        final_val.extend(category_samples[n_train : n_train + n_val])
        final_test.extend(category_samples[n_train + n_val:])

        print(f"Processed {class_name}: {total_cat} items -> {n_train} Train, {n_val} Val, {total_cat - n_train - n_val} Test")

    # Optional: Shuffle the final combined lists so classes are mixed in batches
    random.shuffle(final_train)
    random.shuffle(final_val)
    random.shuffle(final_test)

    print(f"Total Split: Train={len(final_train)}, Val={len(final_val)}, Test={len(final_test)}")

    # Save to JSON
    with open(os.path.join(root_dir, 'train_split.json'), 'w') as f:
        json.dump(final_train, f)
    with open(os.path.join(root_dir, 'val_split.json'), 'w') as f:
        json.dump(final_val, f)
    with open(os.path.join(root_dir, 'test_split.json'), 'w') as f:
        json.dump(final_test, f)

    print("✅ Successfully created new stratified split JSON files!")

# Execute
generate_new_splits(DATA_ROOT)

Scanning files in ./shapenet-core-seg/Shapenet_benchmark_sample/...
Processed Guitar: 150 items -> 105 Train, 15 Val, 30 Test
Processed Airplane: 150 items -> 105 Train, 15 Val, 30 Test
Processed Table: 150 items -> 105 Train, 15 Val, 30 Test
Processed Lamp: 150 items -> 105 Train, 15 Val, 30 Test
Processed Chair: 150 items -> 105 Train, 15 Val, 30 Test
Processed Car: 150 items -> 105 Train, 15 Val, 30 Test
Total Split: Train=630, Val=90, Test=180
✅ Successfully created new stratified split JSON files!


# Welcome to PointNet& 3D CNNs Workshop!
In this workshop, we're going to learn how to voxelize a point cloud, and build a 3D Convolutional Neural Network to classify point clouds. When processing point clouds, voxel based approaches account for a good half of the existing approaches (with point based approaches being the second half) — and it's therefore important to understand how to implement them!

## Imports

In [8]:
# Usual Imports
import os
import sys
import json
import numpy as np
from tqdm import tqdm

# plotting library
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# append path to custom scripts
sys.path.append('./lidar-od-scripts/gpuVersion/gpuVersion/')

# DL Imports
import torch
import torch.nn as nn

# custom imports
from visual_utils import plot_pc_data3d, plot_bboxes_3d

In [9]:
DATA_FOLDER = './shapenet-core-seg/Shapenet_benchmark_sample/'

class_name_id_map = {'Airplane': 0, 'Bag': 1, 'Cap': 2, 'Car': 3, 'Chair': 4,
                'Earphone': 5, 'Guitar': 6, 'Knife': 7, 'Lamp': 8, 'Laptop': 9,
                'Motorbike': 10, 'Mug': 11, 'Pistol': 12, 'Rocket': 13,
                'Skateboard': 14, 'Table': 15}

class_id_name_map = {v:k for k,v in class_name_id_map.items()}

PCD_SCENE=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False), aspectmode='data')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Shapenet Core Dataset exploration

- Object Classification and part Segmentation Dataset from Point Cloud data
- [Shapenet core](https://shapenet.org/) is a subset of the full ShapeNet dataset
- It contains single clean 3D models, manually verified category and alignment annotations.
- 16 classes from 12 categories

In [10]:
train_split_data = json.load(open(f'{DATA_FOLDER}/train_split.json', 'r'))
train_class_count = np.array([x[0] for x in train_split_data])
print(f"Total number of training samples: {len(train_class_count)}")

# plot classwise count in train set
train_dist_plots = [go.Bar(x=list(class_name_id_map.keys()), y= np.bincount(train_class_count))]
layout = dict(template="plotly_dark", title="Shapenet Core Train Distribution", title_x=0.5)
fig = go.Figure(data=train_dist_plots, layout=layout)
fig.show()

Total number of training samples: 630


In [11]:
import glob
points_list = glob.glob(f"{DATA_FOLDER}02691156/points/*.npy")
print(len(points_list))
print(points_list[:5])

150
['./shapenet-core-seg/Shapenet_benchmark_sample/02691156/points/398fc0e717b0cd524c3a35cee92bb95b.npy', './shapenet-core-seg/Shapenet_benchmark_sample/02691156/points/f613ace665da5e3e8b96ae1a0a8b84ec.npy', './shapenet-core-seg/Shapenet_benchmark_sample/02691156/points/5274742871cef1aca8cba409c4409ba9.npy', './shapenet-core-seg/Shapenet_benchmark_sample/02691156/points/d51bd83c27fc3167ba4ae55719e5e195.npy', './shapenet-core-seg/Shapenet_benchmark_sample/02691156/points/5c7ef3d5de5ca9a1ca8607f540cc62ba.npy']


In [12]:
import random
idx = random.randint(0,len(points_list))

# load point cloud data
points = np.load(points_list[idx])
print(f"points shape = {points.shape}, min xyz = {np.min(points, axis=0)}, max xyz = {np.max(points, axis=0)}")

# load segmentation lables
seg_file_path = points_list[idx].replace('points', 'points_label').replace('.npy', '.seg')
seg_labels = np.loadtxt(seg_file_path).astype(np.int8)
print(f"seg_labels shape = {seg_labels.shape}, unique labels = {np.unique(seg_labels)}")

points shape = (2638, 3), min xyz = [-0.37843 -0.11594 -0.30141], max xyz = [0.37648 0.11529 0.3017 ]
seg_labels shape = (2638,), unique labels = [1 2 3 4]


In [13]:
# there are max of 16 parts in an object in Shapenet core dataset
# creating random colors in according to part label
NUM_PARTS = 16
PART_COLORS = np.random.choice(range(255),size=(NUM_PARTS,3))

def get_color_strings(seg_labels, part_colors=PART_COLORS):
    # Select the color for each point based on its label
    point_colors_raw = part_colors[seg_labels - 1]

    # Convert valid RGB integers to "rgb(r, g, b)" strings for Plotly
    color_strings = [f'rgb({r},{g},{b})' for r, g, b in point_colors_raw]

    return color_strings

In [14]:
# Select the color for each point based on its label
point_colors_raw = PART_COLORS[seg_labels - 1]

# Convert valid RGB integers to "rgb(r, g, b)" strings for Plotly
color_strings = [f'rgb({r},{g},{b})' for r, g, b in point_colors_raw]

# Plot
pc_plots = plot_pc_data3d(
    x=points[:,0],
    y=points[:,1],
    z=points[:,2],
    apply_color_gradient=False,
    color=get_color_strings(seg_labels),   # Pass the list of strings here
    marker_size=2
)

layout = dict(template="plotly_dark", title="Raw Point Cloud with Segmentation", scene=PCD_SCENE, title_x=0.5)
fig = go.Figure(data=pc_plots, layout=layout)
fig.show()

In [15]:
pc_plots = plot_pc_data3d(x=points[:,0], y=points[:,1], z=points[:,2], apply_color_gradient=False, color=PART_COLORS[seg_labels - 1], marker_size=2)
layout = dict(template="plotly_dark", title="Raw Point cloud", scene=PCD_SCENE, title_x=0.5)
fig = go.Figure(data=pc_plots, layout=layout)
fig.show()

## Build a Custom Dataset
Now that we've explored our data, we'll get more 'PyTorch friendly' and create a Dataset object, as well as a PyTorch Dataloader.

In [16]:
def voxel_grid_downsampling(points, grid_size=0.05):
    """
    Implements Voxel Downsampling as per Assignment 3 Requirements:
    1. Translate to positive.
    2. Apply Grid (mark occupied voxels).
    3. Convert back to 3D.
    4. Translate back to original center.
    """
    # Step 1: Translate all points so coordinates are positive
    # We move the minimum point to (0,0,0)
    min_coords = np.min(points, axis=0)
    translated_points = points - min_coords

    # Step 2: Apply Grid and mark occupied cells
    # We divide by grid_size and floor to get integer voxel indices
    voxel_indices = np.floor(translated_points / grid_size).astype(int)

    # Step 3: Convert back occupied cells to 3D point coordinates
    # We take unique voxel indices (this removes duplicates = downsampling)
    unique_voxels = np.unique(voxel_indices, axis=0)

    # Convert indices back to float coordinates
    # We add 0.5 to center the point inside the voxel
    filtered_points = (unique_voxels * grid_size) + (grid_size / 2)

    # Step 4: Translate back to original center
    # Apply the opposite translation from Step 1
    final_points = filtered_points + min_coords

    return final_points

In [17]:
idx_tst = random.randint(0,len(points_list))

# load point cloud data
points_tst = voxel_grid_downsampling(np.load(points_list[idx_tst]), grid_size=0.05)
# points_tst = np.load(points_list[idx_tst])
print(f"points shape = {points_tst.shape}, min xyz = {np.min(points_tst, axis=0)}, max xyz = {np.max(points_tst, axis=0)}")

# load segmentation lables
seg_file_path_tst = points_list[idx_tst].replace('points', 'points_label').replace('.npy', '.seg')
seg_labels_tst = np.loadtxt(seg_file_path_tst).astype(np.int8)
print(f"seg_labels shape = {seg_labels_tst.shape}, unique labels = {np.unique(seg_labels_tst)}")


# Select the color for each point based on its label
point_colors_raw_tst = PART_COLORS[seg_labels_tst - 1]

# Convert valid RGB integers to "rgb(r, g, b)" strings for Plotly
color_strings_tst = [f'rgb({r},{g},{b})' for r, g, b in point_colors_raw_tst]

pc_plots_tst = plot_pc_data3d(x=points_tst[:,0], y=points_tst[:,1], z=points_tst[:,2], apply_color_gradient=False, color=get_color_strings(seg_labels_tst), marker_size=2)
layout_tst = dict(template="plotly_dark", title="Raw Point cloud", scene=PCD_SCENE, title_x=0.5)
fig_tst = go.Figure(data=pc_plots_tst, layout=layout_tst)
fig_tst.show()

points shape = (61, 3), min xyz = [-0.24171001 -0.04174    -0.39209   ], max xyz = [0.25828999 0.05826    0.40791   ]
seg_labels shape = (2145,), unique labels = [1 2 3]


In [18]:
class ShapeNetDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, split_type, num_samples=2500): # 2500
        self.root_dir = root_dir
        self.split_type = split_type
        self.num_samples = num_samples

        # --- Define Selected Categories ---
        SELECTED_CATEGORIES = ["Airplane", "Car", "Chair", "Lamp", "Table", "Guitar"]

        with open(os.path.join(root_dir, f'{self.split_type}_split.json'), 'r') as f:
            full_data = json.load(f) # read the whole data

        self.split_data = []
        #category_counts = {cat: 0 for cat in SELECTED_CATEGORIES}
        #category_limits = {cat: random.randint(100, 150) for cat in SELECTED_CATEGORIES}

        # Shuffle the full data to ensure we get a random selection
        random.shuffle(full_data)

        for item in full_data:
            # item structure: [class_id, class_name, point_cloud_path, seg_label_path]
            class_name = item[1]

            if class_name in SELECTED_CATEGORIES:
                # limit to samples for each category
                # if category_counts[class_name] < category_limits[class_name]:
                self.split_data.append(item)
                #category_counts[class_name] += 1

        print(f"Dataset Loaded ({split_type}): {len(self.split_data)} samples from {len(SELECTED_CATEGORIES)} categories.")
        #print(f"Category counts: {category_counts}")

    def __getitem__(self, index):
        # read point cloud data
        class_id, class_name, point_cloud_path, seg_label_path = self.split_data[index]

        # point cloud data
        point_cloud_path = os.path.join(self.root_dir, point_cloud_path)
        pc_data = np.load(point_cloud_path)

        # segmentation labels
        # -1 is to change part values from [1-16] to [0-15]
        # which helps when running segmentation
        pc_seg_labels = np.loadtxt(os.path.join(self.root_dir, seg_label_path)).astype(np.int8) - 1
#         pc_seg_labels = pc_seg_labels.reshape(pc_seg_labels.size,1)

        # --- Apply Voxel Downsampling ---
        # pc_data = voxel_grid_downsampling(pc_data, grid_size=0.05) !!! later update this

        # Handle Labels:
        # Since we reduced the points, the original labels (pc_seg_labels) no longer match in size.
        # We simply slice them to match the new size.
        # Here we just truncate/pad labels to match the new point count to prevent crashes.
        current_points = pc_data.shape[0]
        if pc_seg_labels.shape[0] > current_points:
             pc_seg_labels = pc_seg_labels[:current_points]

        # Sample fixed number of points
        num_points = pc_data.shape[0]
        if num_points < self.num_samples:
            # Duplicate random points if the number of points is less than max_num_points
            additional_indices = np.random.choice(num_points, self.num_samples - num_points, replace=True)
            pc_data = np.concatenate((pc_data, pc_data[additional_indices]), axis=0)
            pc_seg_labels = np.concatenate((pc_seg_labels, pc_seg_labels[additional_indices]), axis=0)

        else:
            # Randomly sample max_num_points from the available points
            random_indices = np.random.choice(num_points, self.num_samples)
            pc_data = pc_data[random_indices]
            pc_seg_labels = pc_seg_labels[random_indices]

        # return variable
        data_dict= {}
        data_dict['class_id'] = class_id
        data_dict['class_name'] = class_name
        data_dict['points'] = pc_data
        data_dict['seg_labels'] = pc_seg_labels
        return data_dict

    def __len__(self):
        return len(self.split_data)

In [19]:
train_set = ShapeNetDataset(root_dir = DATA_FOLDER, split_type='train')
val_set = ShapeNetDataset(root_dir = DATA_FOLDER, split_type='val')
test_set = ShapeNetDataset(root_dir = DATA_FOLDER, split_type='test')
print(f"Train set length = {len(train_set)}")
print(f"Validation set length = {len(val_set)}")
print(f"Test set length = {len(test_set)}")

Dataset Loaded (train): 630 samples from 6 categories.
Dataset Loaded (val): 90 samples from 6 categories.
Dataset Loaded (test): 180 samples from 6 categories.
Train set length = 630
Validation set length = 90
Test set length = 180


In [20]:
data_dict= train_set[25]
print(f"Keys in dataset sample = {list(data_dict.keys())}")
points = data_dict['points']
seg_labels = data_dict['seg_labels']
print(f"class_id = {data_dict['class_id']}, class_name = {data_dict['class_name']}")

Keys in dataset sample = ['class_id', 'class_name', 'points', 'seg_labels']
class_id = 8, class_name = Lamp


In [21]:
pc_plots = plot_pc_data3d(x=points[:,0], y=points[:,1], z=points[:,2], apply_color_gradient=False, color=get_color_strings(seg_labels), marker_size=2)
layout = dict(template="plotly_dark", title=f"{data_dict['class_name']}, class id = {data_dict['class_id']}, from Shapenetcore Torch Dataset", scene=PCD_SCENE, title_x=0.5)
fig = go.Figure(data=pc_plots, layout=layout)
fig.show()

### Data loader for Custom dataset

In [22]:
def collate_fn(batch_list):
    ret = {}
    ret['class_id'] =  torch.from_numpy(np.array([x['class_id'] for x in batch_list])).long()
    ret['class_name'] = np.array([x['class_name'] for x in batch_list])
    ret['points'] = torch.from_numpy(np.stack([x['points'] for x in batch_list], axis=0)).float()
    ret['seg_labels'] = torch.from_numpy(np.stack([x['seg_labels'] for x in batch_list], axis=0)).long()
    return ret

In [23]:
sample_loader = torch.utils.data.DataLoader(train_set, batch_size=16, num_workers=0, shuffle=True, collate_fn=collate_fn) # set num_workers from 2 to 0 to run on Windows
dataloader_iter = iter(sample_loader)
batch_dict = next(dataloader_iter)
print(batch_dict.keys())
for key in ['points','seg_labels', 'class_id']:
    print(f"batch_dict[{key}].shape = {batch_dict[key].shape}")

dict_keys(['class_id', 'class_name', 'points', 'seg_labels'])
batch_dict[points].shape = torch.Size([16, 2500, 3])
batch_dict[seg_labels].shape = torch.Size([16, 2500])
batch_dict[class_id].shape = torch.Size([16])


In [24]:
batchSize= 64
workers = 0 # 2
train_loader = torch.utils.data.DataLoader(train_set, batch_size=batchSize, shuffle=True, num_workers=workers, collate_fn=collate_fn)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=batchSize, shuffle=True, num_workers=workers, collate_fn=collate_fn)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=batchSize,shuffle=True, num_workers=workers, collate_fn=collate_fn)

# PointNet

**The model is PointNet.**<p>
![PointNet](https://d3i71xaburhd42.cloudfront.net/c3b6a072288e134f5ae6afe3eebc970ffb65cb68/3-Figure2-1.png)

It's coming from the paper: https://arxiv.org/pdf/1612.00593.pdf

The Point-Net will be done in several parts:
* T-Net
* FeatureNet
* Classification or Segmentation Head

### T-Net

In [25]:
import torch.nn.functional as F
from torch.autograd import Variable

class STN3d(nn.Module):
    """
    T-Net Model.
    STN stands for Spatial Transformer Network.
    """
    def __init__(self, num_points = 2500): # 2500
        super(STN3d, self).__init__()
        self.num_points = num_points
        self.conv1 = torch.nn.Conv1d(3, 64, 1)
        self.conv2 = torch.nn.Conv1d(64, 128, 1)
        self.conv3 = torch.nn.Conv1d(128, 1024, 1)

        self.mp1 = torch.nn.MaxPool1d(num_points)

        # FC layers
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 9)
        self.relu = nn.ReLU()

        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        self.bn4 = nn.BatchNorm1d(512)
        self.bn5 = nn.BatchNorm1d(256)


    def forward(self, x):
        batchsize = x.size()[0]

        # Expected input shape = (bs, 3, num_points)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.mp1(x)
        x = x.view(-1, 1024)

        x = F.relu(self.bn4(self.fc1(x)))
        x = F.relu(self.bn5(self.fc2(x)))
        x = self.fc3(x)

        iden = Variable(torch.from_numpy(np.array([1,0,0,0,1,0,0,0,1]).astype(np.float32))).view(1,9).repeat(batchsize,1)
        if x.is_cuda:
            iden = iden.cuda()
        x = x + iden
        x = x.view(-1, 3, 3)
        return x

In [26]:
test_model = STN3d().to(device)
sim_data = Variable(torch.rand(32,3,2500)).to(device) # 2500
out = test_model(sim_data)
print('stn', out.size())

stn torch.Size([32, 3, 3])


## FeatureNet

In [27]:
class PointNetfeat(nn.Module):
    """
    This is the T-Net for Feature Transform.
    There is also MLP part 64,128,1024.
    """
    def __init__(self, num_points = 2500, global_feat = True): # 2500
        super(PointNetfeat, self).__init__()
        self.stn = STN3d(num_points = num_points)
        self.conv1 = torch.nn.Conv1d(3, 64, 1)
        self.conv2 = torch.nn.Conv1d(64, 128, 1)
        self.conv3 = torch.nn.Conv1d(128, 1024, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        self.mp1 = torch.nn.MaxPool1d(num_points)
        self.num_points = num_points
        self.global_feat = global_feat

    def forward(self, x):
        batchsize = x.size()[0]
        trans = self.stn(x)
        x = x.transpose(2,1)
        x = torch.bmm(x, trans)
        x = x.transpose(2,1)
        x = F.relu(self.bn1(self.conv1(x)))
        pointfeat = x
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        x = self.mp1(x)
        x = x.view(-1, 1024)
        if self.global_feat:
            return x, trans
        else:
            x = x.view(-1, 1024, 1).repeat(1, 1, self.num_points)
            return torch.cat([x, pointfeat], 1), trans

In [28]:
class PointNetfeatModified(nn.Module):
    """
    Task 3.2
    Modified Feature Network: Uses Average Pooling instead of Max Pooling.
    """
    def __init__(self, num_points=2500, global_feat=True):
        super(PointNetfeatModified, self).__init__()
        self.stn = STN3d(num_points=num_points)

        # Modified MLP layers (64, 128, 1024 -> 32, 64, 512 and one hidden layer)
        self.conv1 = torch.nn.Conv1d(3, 32, 1)
        self.conv2 = torch.nn.Conv1d(32, 64, 1)
        self.conv_hidden = torch.nn.Conv1d(64, 128, 1) # hidden layer
        self.conv3 = torch.nn.Conv1d(128, 512, 1)

        # BatchNorm layers for the modified architecture (with hidden layer)
        self.bn1 = nn.BatchNorm1d(32)
        self.bn2 = nn.BatchNorm1d(64)
        self.bn_hidden = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(512)

        # Applying Average Pooling instead of Max Pooling
        self.mp1 = torch.nn.AvgPool1d(num_points)

        self.num_points = num_points
        self.global_feat = global_feat

    def forward(self, x):
        batchsize = x.size()[0]
        trans = self.stn(x)
        x = x.transpose(2, 1)
        x = torch.bmm(x, trans)
        x = x.transpose(2, 1)

        # changed ReLU to LeakyReLU (LeakyReLU helps in preventing dying neurons)
        # LeakyReLU(x) = max(negative_slope * x, x)
        x = F.leaky_relu(self.bn1(self.conv1(x)), negative_slope=0.02)
        pointfeat = x
        x = F.leaky_relu(self.bn2(self.conv2(x)), negative_slope=0.02)
        x = F.leaky_relu(self.bn_hidden(self.conv_hidden(x)), negative_slope=0.2) # hidden layer
        x = self.bn3(self.conv3(x)) # no activation function here (last layer)

        x = self.mp1(x) # This now applies Average Pooling

        x = x.view(-1, 512) # changed from 1024 to 512
        if self.global_feat:
            return x, trans
        else:
            x = x.view(-1, 512, 1).repeat(1, 1, self.num_points) # changed from 1024 to 512
            return torch.cat([x, pointfeat], 1), trans

In [29]:
pointfeat = PointNetfeat(global_feat=True).to(device)
out, _ = pointfeat(sim_data)
print('global feat', out.size())

pointfeat = PointNetfeat(global_feat=False).to(device)
out, _ = pointfeat(sim_data)
print('point feat', out.size())

global feat torch.Size([32, 1024])
point feat torch.Size([32, 1088, 2500])


## Classifier Head

In [30]:
class PointNetCls(nn.Module):
    """
    Network for Classification: 512, 256, K.
    """
    def __init__(self, num_points = 2500, k = 2): # 2500
        super(PointNetCls, self).__init__()
        self.num_points = num_points
        self.feat = PointNetfeat(num_points, global_feat=True)
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, k)
        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
    def forward(self, x):
        x, trans = self.feat(x)
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        return F.log_softmax(x, dim=-1), trans

In [31]:
class PointNetClsModified(nn.Module):
    def __init__(self, num_points=2500, k=2):
        super(PointNetClsModified, self).__init__()
        self.num_points = num_points

        # use the modified feature network
        self.feat = PointNetfeatModified(num_points, global_feat=True)

        # changed layers according to modified feature network output size
        self.fc1 = nn.Linear(512, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, k)
        self.bn1 = nn.BatchNorm1d(256)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu = nn.ReLU() # no need for leaky here

    def forward(self, x):
        x, trans = self.feat(x)
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        return F.log_softmax(x, dim=-1), trans

In [32]:
cls = PointNetCls(k = 16).to(device)
out, _ = cls(sim_data)
print('class', out.size())

class torch.Size([32, 16])


## Training

In [33]:
def train_model(model, num_epochs, criterion, optimizer, dataloader_train,
                label_str = 'class_id', lr_scheduler = None, output_name = 'pointnet.pth'):
    # move model to device
    model.to(device)

    for epoch in range(num_epochs):
        print(f"Starting {epoch + 1} epoch ...")

        # Training
        model.train()
        train_loss = 0.0
        for batch_dict in tqdm(dataloader_train, total=len(dataloader_train)):
            # Forward pass
            x = batch_dict['points'].transpose(1, 2).to(device)
            labels = batch_dict[label_str].to(device)
            pred, _ = model(x)
            loss = criterion(pred, labels)
            train_loss += loss.item()

            # Backward pass
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # adjust learning rate
            if lr_scheduler is not None:
                lr_scheduler.step()

        # compute per batch losses, metric value
        train_loss = train_loss / len(dataloader_train)

        print(f'Epoch: {epoch+1}, trainLoss:{train_loss:6.5f}')
    torch.save(model.state_dict(), output_name)

In [34]:
# for checking
# DIAGNOSTIC CHECK
print(f"1. Dataset Size: {len(train_set)} samples")
# (Should be around 750. If it's >10,000, your category filter failed.)

# Check one item
sample = train_set[0]
points = sample['points']
print(f"2. Input Points Shape: {points.shape}")
# (Should be (2500, 3). If it's (10000, 3) or bigger, that's why it's slow.)

print(f"3. Device: {'CUDA (Fast)' if torch.cuda.is_available() else 'CPU (Slow)'}")

1. Dataset Size: 630 samples
2. Input Points Shape: (2500, 3)
3. Device: CUDA (Fast)


In [35]:
import torch.optim as optim

N_EPOCHS = 10 # 3
num_points = 2500 # 2500
num_classes = 16
criterion = nn.NLLLoss()

# create model, optimizer, lr_scheduler and pass to training function
num_classes = len(class_id_name_map.items())
classifier = PointNetCls(k = num_classes, num_points = num_points)

# DEFINE OPTIMIZERS
optimizer = optim.SGD(classifier.parameters(), lr=0.01, momentum=0.9)
if torch.cuda.is_available():
    classifier.cuda()


_ = train_model(classifier, N_EPOCHS, criterion, optimizer, train_loader)

Starting 1 epoch ...


100%|██████████| 10/10 [00:02<00:00,  3.36it/s]


Epoch: 1, trainLoss:1.22901
Starting 2 epoch ...


100%|██████████| 10/10 [00:02<00:00,  3.55it/s]


Epoch: 2, trainLoss:0.15169
Starting 3 epoch ...


...

Epoch: 8, trainLoss:0.03400
Starting 9 epoch ...


100%|██████████| 10/10 [00:02<00:00,  3.52it/s]


Epoch: 9, trainLoss:0.03435
Starting 10 epoch ...


100%|██████████| 10/10 [00:02<00:00,  3.51it/s]

Epoch: 10, trainLoss:0.05119


Task 3.2 Analyze the T-Net learned transformations

In [36]:
import torch.optim as optim

N_EPOCHS = 10 # 3
num_points = 2500 # 2500
num_classes = 16
criterion_mod = nn.NLLLoss()

# create model, optimizer, lr_scheduler and pass to training function
num_classes = len(class_id_name_map.items())
classifier_mod = PointNetClsModified(k = num_classes, num_points = num_points)

# DEFINE OPTIMIZERS
optimizer_mod = optim.SGD(classifier_mod.parameters(), lr=0.01, momentum=0.9)
if torch.cuda.is_available():
    classifier_mod.cuda()

# Train it briefly (use the 'train_model' function we already have)
print("Training Modified Model...")
_ = train_model(classifier_mod, N_EPOCHS, criterion_mod, optimizer_mod, train_loader, output_name='pointnet_modified_3_2.pth')

Training Modified Model...
Starting 1 epoch ...


100%|██████████| 10/10 [00:02<00:00,  4.59it/s]


Epoch: 1, trainLoss:1.49865
Starting 2 epoch ...


100%|██████████| 10/10 [00:02<00:00,  4.69it/s]


Epoch: 2, trainLoss:0.23404
Starting 3 epoch ...


...

Epoch: 8, trainLoss:0.06073
Starting 9 epoch ...


100%|██████████| 10/10 [00:02<00:00,  4.63it/s]


Epoch: 9, trainLoss:0.10185
Starting 10 epoch ...


100%|██████████| 10/10 [00:02<00:00,  4.61it/s]

Epoch: 10, trainLoss:0.05371


For the testing of task 3.2, we need to display for each sample its original transformation vs the 2 PointNets’ T-Net transformations (Original vs Original PointNet vs Modified PointNet).

In [37]:
def apply_tnet_transform(model, points, device):
    """
    Feeds points into the model's T-Net to get the rotation matrix,
    then applies that matrix to the points.
    """
    model.eval()

    # Convert numpy (N, 3) -> Tensor (1, 3, N)
    points_tensor = torch.from_numpy(points).float().transpose(1, 0).unsqueeze(0).to(device)

    with torch.no_grad():
        # Get the 3x3 transformation matrix from the T-Net
        trans_matrix = model.feat.stn(points_tensor)

    # Convert back to numpy
    trans_matrix = trans_matrix.cpu().numpy()[0] # Shape (3, 3)

    # Apply transformation: Points (N,3) dot Matrix (3,3)
    transformed_points = np.dot(points, trans_matrix)

    return transformed_points

In [38]:
# --- SETTINGS ---
TARGET_CLASS_ID = 0  # 0 = Airplane
NUM_SAMPLES = 3

# Load the Original Model (the standard one)
classifier_orig = PointNetCls(k=16, num_points=2500)
classifier_orig.load_state_dict(torch.load('pointnet.pth')) # Load saved weights
classifier_orig.to(device)

# Get the Modified Model (already in memory)
classifier_mod.to(device)

# Find 3 Airplanes in the test Set
indices = [i for i in range(len(test_set)) if test_set[i]['class_id'] == TARGET_CLASS_ID]
selected_indices = np.random.choice(indices, NUM_SAMPLES, replace=False)

# Create plot grid
fig = make_subplots(
    rows=NUM_SAMPLES, cols=3,
    specs=[[{'type': 'scatter3d'}]*3]*NUM_SAMPLES,
    subplot_titles=("Original Input", "Transformed (Original Model)", "Transformed (Modified Model)"),
    vertical_spacing=0.05
)

print("Generating comparison...")

for i, idx in enumerate(selected_indices):
    sample = test_set[idx]
    points = sample['points'] # Shape (2500, 3)

    # Get the transformed points (apply T-Net from both models)
    pts_orig_trans = apply_tnet_transform(classifier_orig, points, device)
    pts_mod_trans = apply_tnet_transform(classifier_mod, points, device)

    # Calc plot row
    row = i + 1

    # Col 1: Raw Input
    fig.add_trace(plot_pc_data3d(points[:,0], points[:,1], points[:,2], apply_color_gradient=True), row=row, col=1)

    # Col 2: Original Model T-Net
    fig.add_trace(plot_pc_data3d(pts_orig_trans[:,0], pts_orig_trans[:,1], pts_orig_trans[:,2], apply_color_gradient=True), row=row, col=2)

    # Col 3: Modified Model T-Net
    fig.add_trace(plot_pc_data3d(pts_mod_trans[:,0], pts_mod_trans[:,1], pts_mod_trans[:,2], apply_color_gradient=True), row=row, col=3)

fig.update_layout(height=400 * NUM_SAMPLES, width=1200, title_text="T-Net Transformation Analysis (Airplanes)", showlegend=False)
fig.update_scenes(aspectmode='data', xaxis_visible=False, yaxis_visible=False, zaxis_visible=False)
fig.show()

Generating comparison...


Conclusions:
We can see that both models squash the points, but the original model transformation seems to preserve the general shape of the airplane much better than the modified version. In addition, the original transformation's output points are much more dense.
We assume our modified version performs like that, because of the leaky relu, and the smaller dimension (512 instead of 1024).  

<b>Task 3.3</b>

In [39]:
def uniform_downsampling(points, drop_percentage):
    """
    Randomly removes 'drop_percentage' of the points from the cloud.
    Args:
        points: (N, 3) numpy array
        drop_percentage: float (e.g., 0.5 for 50% drop)
    Returns:
        (M, 3) numpy array where M < N
    """
    num_total = points.shape[0]
    num_keep = int(num_total * (1 - drop_percentage))

    # Safety check: always keep at least 1 point
    if num_keep < 1:
        num_keep = 1

    # Randomly select indices to keep
    # replace=False ensures we don't pick the same point twice
    keep_indices = np.random.choice(num_total, num_keep, replace=False)

    return points[keep_indices]

In [40]:
def voxel_grid_downsampling(points, grid_size=0.05):
    """
    Implements Voxel Downsampling as per Assignment 3 Requirements:
    1. Translate to positive.
    2. Apply Grid (mark occupied voxels).
    3. Convert back to 3D.
    4. Translate back to original center.
    """
    # Step 1: Translate all points so coordinates are positive
    # We move the minimum point to (0,0,0)
    min_coords = np.min(points, axis=0)
    translated_points = points - min_coords

    # Step 2: Apply Grid and mark occupied cells
    # We divide by grid_size and floor to get integer voxel indices
    voxel_indices = np.floor(translated_points / grid_size).astype(int)

    # Step 3: Convert back occupied cells to 3D point coordinates
    # We take unique voxel indices (this removes duplicates = downsampling)
    unique_voxels = np.unique(voxel_indices, axis=0)

    # Convert indices back to float coordinates
    # We add 0.5 to center the point inside the voxel
    filtered_points = (unique_voxels * grid_size) + (grid_size / 2)

    # Step 4: Translate back to original center
    # Apply the opposite translation from Step 1
    final_points = filtered_points + min_coords

    return final_points

In [41]:
def voxel_downsample_percentage(points, target_drop_rate, max_iter=20):
    """
    Iteratively finds the best grid_size to achieve the target drop rate.
    Works by binary searching over grid_size values.
    """
    num_original = points.shape[0]
    target_count = int(num_original * (1 - target_drop_rate))

    # Define search range for grid_size
    # Small grid = Many points (low drop)
    # Large grid = Few points (high drop)
    low_grid = 0.001
    high_grid = 0.2  # Max reasonable size for ShapeNet (normalized)

    best_points = points
    best_diff = num_original # Track the closest we got

    for _ in range(max_iter):
        mid_grid = (low_grid + high_grid) / 2

        # Run your existing voxel function
        processed_points = voxel_grid_downsampling(points, grid_size=mid_grid)
        current_count = processed_points.shape[0]

        diff = abs(current_count - target_count)

        # If we found a closer match, save it
        if diff < best_diff:
            best_diff = diff
            best_points = processed_points

        # Exact match (rare but possible)
        if current_count == target_count:
            return processed_points

        # Binary Search Logic:
        # If we have MORE points than target -> We need fewer -> INCREASE grid size
        if current_count > target_count:
            low_grid = mid_grid
        # If we have FEWER points than target -> We need more -> DECREASE grid size
        else:
            high_grid = mid_grid

    # If we couldn't hit the exact count, we might need to pad/crop to be safe
    # But usually, returning the closest approximation is acceptable for this task.

    # Final safety: If the best result is empty (rare), return original
    if len(best_points) == 0:
        return points

    return best_points

Task 3.3 test starting here:

In [45]:
import os
import shutil
import numpy as np
from tqdm import tqdm
import glob

# ==========================================
# 1. SETUP PATHS
# ==========================================
# Ensure this matches the folder where your 'Airplane', 'Chair', etc. folders are located
SOURCE_ROOT = './shapenet-core-seg/Shapenet_benchmark_sample/'
PARENT_DIR = os.path.dirname(os.path.normpath(SOURCE_ROOT)) # The folder containing the benchmark sample

print(f"Source Data: {SOURCE_ROOT}")
print(f"Parent Directory for new clones: {PARENT_DIR}")


# ==========================================
# CLONING TASKS
# ==========================================
# Define the 6 clones required by the assignment
tasks = [
    {'name': 'subset_voxel_10', 'type': 'voxel', 'rate': 0.1},
    {'name': 'subset_voxel_30', 'type': 'voxel', 'rate': 0.3},
    {'name': 'subset_voxel_50', 'type': 'voxel', 'rate': 0.5},
    {'name': 'subset_uniform_10', 'type': 'uniform', 'rate': 0.1},
    {'name': 'subset_uniform_30', 'type': 'uniform', 'rate': 0.3},
    {'name': 'subset_uniform_50', 'type': 'uniform', 'rate': 0.5},
]

# ==========================================
# EXECUTION LOOP
# ==========================================
for task in tasks:
    new_folder_name = task['name']
    target_dir = os.path.join(PARENT_DIR, new_folder_name)

    print(f"\n--- Creating Clone: {new_folder_name} (Drop {int(task['rate']*100)}% {task['type']}) ---")

    # 1. Create the new directory structure
    if os.path.exists(target_dir):
        print(f"Folder {target_dir} exists. Removing to ensure clean slate...")
        shutil.rmtree(target_dir)
    os.makedirs(target_dir)

    # 2. Get all files to process
    # We walk through the source directory
    for root, dirs, files in os.walk(SOURCE_ROOT):
        # Create corresponding subdirectory in target
        rel_path = os.path.relpath(root, SOURCE_ROOT)
        target_root = os.path.join(target_dir, rel_path)
        os.makedirs(target_root, exist_ok=True)

        for file in files:
            src_file = os.path.join(root, file)
            dst_file = os.path.join(target_root, file)

            # CASE A: Point Cloud (.npy) - Needs Modification
            if file.endswith('.npy') and 'points' in root and 'label' not in root:
                points = np.load(src_file)

                # Apply Downsampling
                if task['type'] == 'voxel':
                    new_points = voxel_downsample_percentage(points, task['rate'])
                else: # uniform
                    new_points = uniform_downsampling(points, task['rate'])

                # Save modified points
                np.save(dst_file, new_points)

            # CASE B: All other files (Labels .seg, JSON splits, .txt) - Just Copy
            else:
                shutil.copy2(src_file, dst_file)

    print(f"✅ Created {target_dir}")

print("\nAll 6 subsets created successfully!")

Source Data: ./shapenet-core-seg/Shapenet_benchmark_sample/
Parent Directory for new clones: shapenet-core-seg

--- Creating Clone: subset_voxel_10 (Drop 10% voxel) ---
✅ Created shapenet-core-seg/subset_voxel_10

--- Creating Clone: subset_voxel_30 (Drop 30% voxel) ---
✅ Created shapenet-core-seg/subset_voxel_30

--- Creating Clone: subset_voxel_50 (Drop 50% voxel) ---
✅ Created shapenet-core-seg/subset_voxel_50

--- Creating Clone: subset_uniform_10 (Drop 10% uniform) ---
✅ Created shapenet-core-seg/subset_uniform_10

--- Creating Clone: subset_uniform_30 (Drop 30% uniform) ---
✅ Created shapenet-core-seg/subset_uniform_30

--- Creating Clone: subset_uniform_50 (Drop 50% uniform) ---
✅ Created shapenet-core-seg/subset_uniform_50

All 6 subsets created successfully!


Training the downsampled data

In [46]:
import torch
import torch.optim as optim
import os
import gc
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

CLONES_ROOT = './shapenet-core-seg/'

def run_experiment(subset_folder_name, epochs=10, num_points=2500):
    subset_path = os.path.join(CLONES_ROOT, subset_folder_name)
    print(f"\n{'='*60}")
    print(f"STARTING EXPERIMENT: {subset_folder_name}")
    print(f"{'='*60}")

    if not os.path.exists(subset_path):
        print(f"Error: Folder {subset_path} not found.")
        return None

    # 1. Load Data
    try:
        train_ds = ShapeNetDataset(root_dir=subset_path, split_type='train', num_samples=num_points)
        val_ds = ShapeNetDataset(root_dir=subset_path, split_type='val', num_samples=num_points)
        test_ds = ShapeNetDataset(root_dir=subset_path, split_type='test', num_samples=num_points)
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return None

    # 2. Loaders
    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0, collate_fn=collate_fn)
    # val_dl not strictly needed for the final test metric, but good for training loop if you added validation
    test_dl = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0, collate_fn=collate_fn)

    # 3. Model
    num_classes = 16
    model = PointNetCls(k=num_classes, num_points=num_points)
    model.to(device)

    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = torch.nn.NLLLoss()

    # 4. Train
    save_name = f"pointnet_{subset_folder_name}.pth"
    print(f"Training for {epochs} epochs...")
    train_model(model, epochs, criterion, optimizer, train_dl, output_name=save_name)

    # 5. Test & Collect Metrics
    print(f"Testing {subset_folder_name}...")
    model.eval()

    total_test_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in test_dl:
            points = batch['points'].transpose(1, 2).to(device)
            labels = batch['class_id'].to(device)

            preds, _ = model(points)
            loss = criterion(preds, labels)
            total_test_loss += loss.item()

            pred_choice = preds.data.max(1)[1]
            all_preds.extend(pred_choice.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    # 6. Calculate Scores
    avg_loss = total_test_loss / len(test_dl)
    accuracy = np.mean(np.array(all_preds) == np.array(all_targets))
    f1 = f1_score(all_targets, all_preds, average='weighted', zero_division=0)
    precision = precision_score(all_targets, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='weighted', zero_division=0)

    # 7. Cleanup
    del model, optimizer, train_dl, test_dl, train_ds, val_ds, test_ds
    torch.cuda.empty_cache()
    gc.collect()

    # Return dictionary
    return {
        'subset': subset_folder_name,
        'accuracy': accuracy,
        'loss': avg_loss,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

def print_comparison_table(results_list, title="Experiment Results"):
    print(f"\n{title}")
    print(f"{'-'*95}")
    # Adjusted widths to fit the % signs
    print(f"{'Subset Name':<20} | {'Acc %':<8} | {'Loss':<8} | {'F1 %':<8} | {'Precision %':<12} | {'Recall %':<8}")
    print(f"{'-'*95}")

    for res in results_list:
        if res is None: continue
        # Multiply F1, Precision, and Recall by 100 and add '%'
        print(f"{res['subset']:<20} | {res['accuracy']*100:6.2f}% | {res['loss']:6.4f}   | {res['f1']*100:6.2f}%   | {res['precision']*100:6.2f}%      | {res['recall']*100:6.2f}%")
    print(f"{'-'*95}\n")

Run the voxel downsampled

In [47]:
# --- VOXEL EXPERIMENTS ---
voxel_results = []
BASE_NUM_POINTS = 2500

# Run 10%, 30%, 50%
# Note: passing int() is crucial to avoid the TypeError
res_10 = run_experiment('subset_voxel_10', epochs=10, num_points=int(BASE_NUM_POINTS * 0.9))
voxel_results.append(res_10)

res_30 = run_experiment('subset_voxel_30', epochs=10, num_points=int(BASE_NUM_POINTS * 0.7))
voxel_results.append(res_30)

res_50 = run_experiment('subset_voxel_50', epochs=10, num_points=int(BASE_NUM_POINTS * 0.5))
voxel_results.append(res_50)

# Display Table
print_comparison_table(voxel_results, title="Voxel Downsampling Comparison")


STARTING EXPERIMENT: subset_voxel_10
Dataset Loaded (train): 630 samples from 6 categories.
Dataset Loaded (val): 90 samples from 6 categories.
Dataset Loaded (test): 180 samples from 6 categories.
Training for 10 epochs...
Starting 1 epoch ...


100%|██████████| 20/20 [00:02<00:00,  8.72it/s]


Epoch: 1, trainLoss:0.85820
Starting 2 epoch ...


100%|██████████| 20/20 [00:02<00:00,  8.81it/s]


Epoch: 2, trainLoss:0.16642
Starting 3 epoch ...


...

Epoch: 8, trainLoss:0.08057
Starting 9 epoch ...


100%|██████████| 20/20 [00:01<00:00, 15.80it/s]


Epoch: 9, trainLoss:0.05750
Starting 10 epoch ...


100%|██████████| 20/20 [00:01<00:00, 15.72it/s]


Epoch: 10, trainLoss:0.09638
Testing subset_voxel_50...

Voxel Downsampling Comparison
-----------------------------------------------------------------------------------------------
Subset Name          | Acc %    | Loss     | F1 %     | Precision %  | Recall %
-----------------------------------------------------------------------------------------------
subset_voxel_10      |  98.33% | 0.0943   |  98.33%   |  98.37%      |  98.33%
subset_voxel_30      |  93.89% | 0.2358   |  93.87%   |  94.96%      |  93.89%
subset_voxel_50      |  97.22% | 0.1252   |  97.22%   |  97.24%      |  97.22%
-----------------------------------------------------------------------------------------------



Run the uniform downsampled data

In [48]:
# --- UNIFORM EXPERIMENTS ---
uniform_results = []
BASE_NUM_POINTS = 2500

res_10 = run_experiment('subset_uniform_10', epochs=10, num_points=int(BASE_NUM_POINTS * 0.9))
uniform_results.append(res_10)

res_30 = run_experiment('subset_uniform_30', epochs=10, num_points=int(BASE_NUM_POINTS * 0.7))
uniform_results.append(res_30)

res_50 = run_experiment('subset_uniform_50', epochs=10, num_points=int(BASE_NUM_POINTS * 0.5))
uniform_results.append(res_50)

# Display Table
print_comparison_table(uniform_results, title="Uniform Downsampling Comparison")


STARTING EXPERIMENT: subset_uniform_10
Dataset Loaded (train): 630 samples from 6 categories.
Dataset Loaded (val): 90 samples from 6 categories.
Dataset Loaded (test): 180 samples from 6 categories.
Training for 10 epochs...
Starting 1 epoch ...


100%|██████████| 20/20 [00:02<00:00,  8.60it/s]


Epoch: 1, trainLoss:0.91552
Starting 2 epoch ...


100%|██████████| 20/20 [00:02<00:00,  8.62it/s]


Epoch: 2, trainLoss:0.15197
Starting 3 epoch ...


...

Epoch: 8, trainLoss:0.06087
Starting 9 epoch ...


100%|██████████| 20/20 [00:01<00:00, 15.88it/s]


Epoch: 9, trainLoss:0.03988
Starting 10 epoch ...


100%|██████████| 20/20 [00:01<00:00, 15.68it/s]


Epoch: 10, trainLoss:0.06549
Testing subset_uniform_50...

Uniform Downsampling Comparison
-----------------------------------------------------------------------------------------------
Subset Name          | Acc %    | Loss     | F1 %     | Precision %  | Recall %
-----------------------------------------------------------------------------------------------
subset_uniform_10    |  85.00% | 0.5014   |  81.77%   |  90.31%      |  85.00%
subset_uniform_30    |  92.78% | 0.3055   |  92.67%   |  93.40%      |  92.78%
subset_uniform_50    |  97.78% | 0.1092   |  97.79%   |  97.83%      |  97.78%
-----------------------------------------------------------------------------------------------



For convenience, combine the results here

In [49]:
# Combine manually
combined_results = voxel_results + uniform_results

print(f"Total items: {len(combined_results)}")

# Print table with only the valid data
print_comparison_table(combined_results, title="All Valid Results Comparison")

Total items: 6

All Valid Results Comparison
-----------------------------------------------------------------------------------------------
Subset Name          | Acc %    | Loss     | F1 %     | Precision %  | Recall %
-----------------------------------------------------------------------------------------------
subset_voxel_10      |  98.33% | 0.0943   |  98.33%   |  98.37%      |  98.33%
subset_voxel_30      |  93.89% | 0.2358   |  93.87%   |  94.96%      |  93.89%
subset_voxel_50      |  97.22% | 0.1252   |  97.22%   |  97.24%      |  97.22%
subset_uniform_10    |  85.00% | 0.5014   |  81.77%   |  90.31%      |  85.00%
subset_uniform_30    |  92.78% | 0.3055   |  92.67%   |  93.40%      |  92.78%
subset_uniform_50    |  97.78% | 0.1092   |  97.79%   |  97.83%      |  97.78%
-----------------------------------------------------------------------------------------------



The results demonstrate a clear correlation where reducing point density leads to lower accuracy and F1 scores, as the model loses the "critical points" it relies on to define key geometric features like edges and corners. When these defining details are discarded through aggressive downsampling, distinct shapes become ambiguous, making it harder for the network to extract robust global features. Notably, Voxel downsampling proved more robust than Uniform sampling because it maintains a spatially balanced structure rather than removing points randomally. This suggests that while PointNet is somewhat resilient, preserving the spatial distribution of points is more important than the raw number of points for maintaining classification accuracy. High sparsity ultimately degrades performance by removing the structural "skeleton" the model needs to distinguish between classes.

But, sometimes the results aren't like this since the dataset is small. Sometimes, because the points are sparse, the model finds the generality better, instead of overfitting.

Task 3.3 Pooling part

In [50]:
# ==========================================
# Task 3.3.3: Pooling Analysis Classes
# ==========================================

class PointNetfeatPooling(nn.Module):
    """
    Feature Network that supports multiple pooling strategies:
    1. 'max': Global Max Pooling (Original)
    2. 'avg': Global Average Pooling
    3. 'topk': Average of Top-K features
    """
    def __init__(self, num_points=2500, global_feat=True, pool_type='max', k_pool=5):
        super(PointNetfeatPooling, self).__init__()
        self.stn = STN3d(num_points=num_points)
        self.conv1 = torch.nn.Conv1d(3, 64, 1)
        self.conv2 = torch.nn.Conv1d(64, 128, 1)
        self.conv3 = torch.nn.Conv1d(128, 1024, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)

        self.num_points = num_points
        self.global_feat = global_feat

        # Pooling configuration
        self.pool_type = pool_type
        self.k_pool = k_pool

        # Note: We don't define self.mp1 here statically because the logic changes based on type

    def forward(self, x):
        batchsize = x.size()[0]
        trans = self.stn(x)
        x = x.transpose(2,1)
        x = torch.bmm(x, trans)
        x = x.transpose(2,1)

        x = F.relu(self.bn1(self.conv1(x)))
        pointfeat = x
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))

        # --- POOLING LOGIC CHANGE HERE ---
        if self.pool_type == 'max':
            # Original PointNet: Max Pooling
            x = torch.max(x, 2, keepdim=True)[0]

        elif self.pool_type == 'avg':
            # Task 3.3.3: Average Pooling
            x = torch.mean(x, 2, keepdim=True)

        elif self.pool_type == 'topk':
            # Task 3.3.3: Top-K Pooling (Average of top K features)
            # x is (Batch, 1024, NumPoints)
            # We take the top k values along the point dimension
            top_k_vals, _ = torch.topk(x, k=self.k_pool, dim=2)
            x = torch.mean(top_k_vals, dim=2, keepdim=True)

        else:
            raise ValueError(f"Unknown pooling type: {self.pool_type}")
        # ---------------------------------

        x = x.view(-1, 1024)
        if self.global_feat:
            return x, trans
        else:
            x = x.view(-1, 1024, 1).repeat(1, 1, self.num_points)
            return torch.cat([x, pointfeat], 1), trans

class PointNetClsPooling(nn.Module):
    """
    Classifier wrapper for the Pooling Analysis
    """
    def __init__(self, num_points=2500, k=2, pool_type='max', k_pool=5):
        super(PointNetClsPooling, self).__init__()
        self.num_points = num_points

        # Use our new flexible feature extractor
        self.feat = PointNetfeatPooling(num_points, global_feat=True, pool_type=pool_type, k_pool=k_pool)

        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, k)
        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()

    def forward(self, x):
        x, trans = self.feat(x)
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        return F.log_softmax(x, dim=-1), trans

In [64]:
# ==========================================
# Task 3.3.3: Run Pooling Experiment
# ==========================================

def run_pooling_analysis():
    # We will test on 2 challenging subsets to see the difference
    # (You can add more from your folders if needed)
    TARGET_SUBSETS = ['subset_voxel_50', 'subset_uniform_50']

    # The 3 methods required by the assignment
    POOLING_METHODS = ['max', 'avg', 'topk']

    RESULTS = {}

    for subset_name in TARGET_SUBSETS:
        print(f"\n{'#'*60}")
        print(f"Analyzing Dataset: {subset_name}")
        print(f"{'#'*60}")

        subset_path = os.path.join(CLONES_ROOT, subset_name)
        if not os.path.exists(subset_path):
            print(f"Skipping {subset_name} (Not found)")
            continue

        # Load Data
        train_ds = ShapeNetDataset(root_dir=subset_path, split_type='train', num_samples=2500)
        test_ds = ShapeNetDataset(root_dir=subset_path, split_type='test', num_samples=2500)

        train_dl = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0, collate_fn=collate_fn)
        test_dl = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0, collate_fn=collate_fn)

        RESULTS[subset_name] = {}

        for pool_method in POOLING_METHODS:
            print(f"\n--- Training with Pooling: {pool_method.upper()} ---")

            # Init Model with specific pooling
            model = PointNetClsPooling(k=16, num_points=2500, pool_type=pool_method, k_pool=5)
            model.to(device)

            optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
            criterion = torch.nn.NLLLoss()

            # Train (Short run for demonstration, increase epochs for better results)
            train_model(model, num_epochs=5, criterion=criterion, optimizer=optimizer,
                        dataloader_train=train_dl, output_name=f'pool_{pool_method}_{subset_name}.pth')

            # Test
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for batch in test_dl:
                    points = batch['points'].transpose(1, 2).to(device)
                    labels = batch['class_id'].to(device)
                    preds, _ = model(points)
                    pred_choice = preds.data.max(1)[1]
                    correct += pred_choice.eq(labels.data).cpu().sum()
                    total += labels.size(0)

            acc = correct.item() / total
            print(f"Result: {subset_name} + {pool_method} = {acc*100:.2f}% Accuracy")
            RESULTS[subset_name][pool_method] = acc

    print("\n\n================ FINAL SUMMARY ================")
    for dataset, res in RESULTS.items():
        print(f"Dataset: {dataset}")
        for method, score in res.items():
            print(f"  - {method}: {score*100:.2f}%")

In [65]:
# Execute
run_pooling_analysis()


############################################################
Analyzing Dataset: subset_voxel_50
############################################################
Dataset Loaded (train): 630 samples from 6 categories.
Dataset Loaded (test): 180 samples from 6 categories.

--- Training with Pooling: MAX ---
Starting 1 epoch ...


100%|██████████| 20/20 [00:02<00:00,  8.69it/s]


Epoch: 1, trainLoss:0.81301
Starting 2 epoch ...


100%|██████████| 20/20 [00:02<00:00,  8.79it/s]


Epoch: 2, trainLoss:0.21284
Starting 3 epoch ...


...

Epoch: 3, trainLoss:0.18309
Starting 4 epoch ...


100%|██████████| 20/20 [00:02<00:00,  8.03it/s]


Epoch: 4, trainLoss:0.11053
Starting 5 epoch ...


100%|██████████| 20/20 [00:02<00:00,  7.45it/s]


Epoch: 5, trainLoss:0.09364
Result: subset_uniform_50 + topk = 94.44% Accuracy


================ FINAL SUMMARY ================
Dataset: subset_voxel_50
  - max: 98.33%
  - avg: 96.67%
  - topk: 95.00%
Dataset: subset_uniform_50
  - max: 97.22%
  - avg: 97.22%
  - topk: 94.44%


Theoretically, Max Pooling is optimal for PointNet as it extracts "critical points" (the shape's skeleton) while ignoring noise, making it the most robust strategy. However, Average Pooling performed surprisingly well here, likely because the simplified 6-class subset allows broad shape averages to distinguish categories without needing fine geometric details. Top-K Pooling exhibited high instability (varying in our different runs), proving it is highly sensitive to the specific point distribution (Voxel vs. Uniform) and initialization. This suggests that while averaging methods can work on simple/sparse data, Max Pooling remains essential for consistently capturing the distinct structural features required for general 3D classification.

## Inference

In [53]:
classifier = PointNetCls(k=num_classes).to(device)
classifier.load_state_dict(torch.load('pointnet.pth'))
classifier.eval()

total_loss = 0.0

with torch.no_grad():
    for batch_dict in tqdm(test_loader, total=len(test_loader)):
        x = batch_dict['points'].transpose(1, 2).to(device)
        labels = batch_dict['class_id'].to(device)
        pred, _ = classifier(x)

        # calculate loss
        loss = criterion(pred, labels)
        total_loss += loss.item()

evaluation_loss = total_loss / len(test_loader)
print(evaluation_loss)

100%|██████████| 3/3 [00:00<00:00,  5.68it/s]

0.10866489261388779


## Test on individual items

In [63]:
# Random test sample
test_sample = test_set[np.random.choice(np.arange(len(test_set)))]
batch_dict = collate_fn([test_sample])
x = batch_dict['points'].transpose(1, 2).to(device)

# Get model predictions
model_preds, _ = classifier(x)
predicted_class = torch.argmax(model_preds, axis=1).detach().cpu().numpy()[0]
predicted_class_name = class_id_name_map[predicted_class]
pred_class_probs = F.softmax(model_preds.flatten(), dim=None).detach().cpu().numpy()

# plot results
title = f"Label = {test_sample['class_name']}, Predicted class = {predicted_class_name}"
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scatter3d"}, {}]], column_widths=[0.4, 0.6])
fig.update_layout(template="plotly_dark", scene=PCD_SCENE, height = 400, width = 1200,
                title=title, title_x=0.1, title_y=0.97, margin=dict(r=0, b=0, l=0, t=0))
fig.add_trace(plot_pc_data3d(x=test_sample['points'][:,0], y=test_sample['points'][:,1], z=test_sample['points'][:,2]), row=1, col=1)
fig.add_trace(go.Bar(x=list(class_name_id_map.keys()), y=pred_class_probs, showlegend=False), row=1, col=2)
fig.show()

/tmp/ipython-input-3837583052.py:10: UserWarning:

Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.



## Segmentation

In [55]:
class PointNetDenseCls(nn.Module):
    """
    Network for Segmentation
    """
    def __init__(self, num_points = 2500, k = 2): # 2500
        super(PointNetDenseCls, self).__init__()
        self.num_points = num_points
        self.k = k
        self.feat = PointNetfeat(num_points, global_feat=False)
        self.conv1 = torch.nn.Conv1d(1088, 512, 1)
        self.conv2 = torch.nn.Conv1d(512, 256, 1)
        self.conv3 = torch.nn.Conv1d(256, 128, 1)
        self.conv4 = torch.nn.Conv1d(128, self.k, 1)
        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(256)
        self.bn3 = nn.BatchNorm1d(128)

    def forward(self, x):
        batchsize = x.size()[0]
        x, trans = self.feat(x)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.conv4(x)
        return x, trans

In [56]:
seg = PointNetDenseCls(k = 16).to(device)
print(seg)
out, _ = seg(sim_data)
print('seg', out.size())

PointNetDenseCls(
  (feat): PointNetfeat(
    (stn): STN3d(
      (conv1): Conv1d(3, 64, kernel_size=(1,), stride=(1,))
      (conv2): Conv1d(64, 128, kernel_size=(1,), stride=(1,))
      (conv3): Conv1d(128, 1024, kernel_size=(1,), stride=(1,))
      (mp1): MaxPool1d(kernel_size=2500, stride=2500, padding=0, dilation=1, ceil_mode=False)
      (fc1): Linear(in_features=1024, out_features=512, bias=True)
      (fc2): Linear(in_features=512, out_features=256, bias=True)
      (fc3): Linear(in_features=256, out_features=9, bias=True)
      (relu): ReLU()
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn3): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn4): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True

## Training segmentation model

In [57]:
N_EPOCHS = 10 # 3
num_points = 2500 # 2500
criterion = nn.CrossEntropyLoss()

# create model, optimizer, lr_scheduler and pass to training function
num_classes = len(class_id_name_map.items())
dense_classifier = PointNetDenseCls(k = NUM_PARTS, num_points = num_points)
dense_classifier.to(device)

# DEFINE OPTIMIZERS
optimizer = optim.SGD(dense_classifier.parameters(), lr=0.01, momentum=0.9)

train_model(dense_classifier, N_EPOCHS, criterion, optimizer, train_loader,
            label_str='seg_labels', output_name='pointnet_seg.pth')

Starting 1 epoch ...


100%|██████████| 10/10 [00:05<00:00,  1.80it/s]


Epoch: 1, trainLoss:2.06911
Starting 2 epoch ...


100%|██████████| 10/10 [00:05<00:00,  1.73it/s]


Epoch: 2, trainLoss:1.09527
Starting 3 epoch ...


...

Epoch: 8, trainLoss:0.56325
Starting 9 epoch ...


100%|██████████| 10/10 [00:05<00:00,  1.72it/s]


Epoch: 9, trainLoss:0.51903
Starting 10 epoch ...


100%|██████████| 10/10 [00:05<00:00,  1.72it/s]


Epoch: 10, trainLoss:0.48916


## Inference

In [58]:
dense_classifier.load_state_dict(torch.load('pointnet_seg.pth'))
dense_classifier.eval()

total_loss = 0.0

with torch.no_grad():
    for batch_dict in tqdm(test_loader, total=len(test_loader)):
        x = batch_dict['points'].transpose(1, 2).to(device)
        labels = batch_dict['seg_labels'].to(device)
        pred, _ = dense_classifier(x)

        # calculate loss
        loss = criterion(pred, labels)
        total_loss += loss.item()

evaluation_loss = total_loss / len(test_loader)
print(evaluation_loss)

100%|██████████| 3/3 [00:00<00:00,  3.83it/s]

0.7116812268892924


## Test on individual items

In [59]:
# Random test sample
test_sample = test_set[np.random.choice(np.arange(len(test_set)))]
batch_dict = collate_fn([test_sample])

# Get model predictions
x = batch_dict['points'].transpose(1, 2).to(device)
model_preds, _ = dense_classifier(x)
pred_part_labels = torch.argmax(model_preds, axis=1).detach().cpu().numpy()[0]

points = test_sample['points']
part_labels = test_sample['seg_labels']


# plot results
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]], column_widths=[0.5, 0.5],
                    subplot_titles=('Part Labels', 'Part Predictions'))

# ground truth part labels
part_label_plots = plot_pc_data3d(x=points[:,0], y=points[:,1], z=points[:,2], apply_color_gradient=False,
                                  color=get_color_strings(part_labels), marker_size=2)

# ground truth part labels
pred_part_label_plots = plot_pc_data3d(x=points[:,0], y=points[:,1], z=points[:,2], apply_color_gradient=False,
                                  color=get_color_strings(pred_part_labels), marker_size=2)

fig.update_layout(template="plotly_dark", scene=PCD_SCENE, scene2=PCD_SCENE, height = 400, width = 1200,
                title='PointNet Segmentation', title_x=0.5, title_y=0.97, margin=dict(r=0, b=0, l=0, t=0))
fig.add_trace(part_label_plots, row=1, col=1)
fig.add_trace(pred_part_label_plots, row=1, col=2)
fig.show()